# Model Adaptation

**Module:** 05 — LLM Fundamentals

Fine-tuning vs PEFT methods—LoRA, QLoRA, and adapters—when each is worth it.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Contrast full fine-tuning vs PEFT
- Explain LoRA update math at a high level
- Describe QLoRA's memory trick
- Choose adaptation vs prompts/RAG


## Fine-tuning

**Definition.** Update model weights on task data so behavior changes persistently.

**Why it matters.** When style/format/domain language must be in the model, not only the prompt.

**How it works.** SFT on curated sets; evaluate generalizations and regressions; version artifacts.

**Intuition.** Sending the intern to a specialized course.

**Common pitfalls.**
- Fine-tuning to memorize facts better solved by RAG
- Tiny dirty datasets

**When to use.** Stable formats, domain jargon, or tones prompts cannot lock in.


In [ ]:
# Demo 1 — decide FT vs RAG
def choose(need_facts, need_style):
    if need_facts and not need_style: return "RAG/prompt"
    if need_style and not need_facts: return "SFT/LoRA"
    return "RAG + light SFT"
print(choose(True, False), choose(False, True), choose(True, True))


In [ ]:
# Demo 2 — dataset row
print({"messages":[{"role":"user","content":"..."},{"role":"assistant","content":"..."}]})


In [ ]:
# Demo 3 — regression suite sizes
print({"format_tests": 50, "safety_tests": 50, "domain_tests": 100})


### Try it yourself — Fine-tuning

1. Pick FT vs RAG for: (1) tone of brand, (2) weekly prices.


## PEFT

**Definition.** **Parameter-Efficient Fine-Tuning** updates a small subset/add-on of parameters.

**Why it matters.** Cuts cost/memory and reduces catastrophic forgetting risk vs full FT.

**How it works.** LoRA/adapters/prefix-tuning/etc. train few params; freeze base.

**Intuition.** Add a slim specialist hat instead of brain surgery.

**Common pitfalls.**
- PEFT on hopeless base models
- No eval still

**When to use.** Default adaptation path for most teams.


In [ ]:
# Demo 1 — trainable fraction
total, lora = 7e9, 20e6
print("trainable %", 100*lora/total)


In [ ]:
# Demo 2 — PEFT menu
print(["LoRA", "QLoRA", "Adapters", "Prefix/Prompt tuning"])


### Try it yourself — PEFT

1. Name two ops benefits of PEFT over full FT.


## LoRA

**Definition.** **LoRA** injects low-rank matrices A,B into frozen weights: W' = W + α·B·A.

**Why it matters.** Strong quality/cost tradeoff; easy to swap adapters per tenant/task.

**How it works.** Train A,B on attention/MLP projections; merge or load at runtime.

**Intuition.** A thin correction layer written in a low-dimensional subspace.

**Common pitfalls.**
- Rank too low for hard tasks
- Adapter proliferation without governance

**When to use.** Most custom SFT today.


In [ ]:
# Demo 1 — low-rank update shapes
d, r = 4096, 8
params = d*r + r*d
print("LoRA params per matrix", params, "vs full", d*d)


In [ ]:
# Demo 2 — merge weights
import numpy as np
W = np.zeros((4,4)); A = np.ones((8,4)); B = np.ones((4,8))*0.01
# note shapes educational
B = np.ones((4,8))*0.01; A = np.ones((8,4))
W2 = W + B @ A
print(W2.shape, np.linalg.matrix_rank(B@A))


In [ ]:
# Demo 3 — multi-adapter routing
print({"tenant": "acme", "adapter": "acme_tone_v3"})


### Try it yourself — LoRA

1. Compute LoRA param count for d=4096,r=16 on 4 matrices.


## QLoRA

**Definition.** **QLoRA** keeps the base model quantized (e.g., 4-bit) while training LoRA adapters in higher precision.

**Why it matters.** Enables fine-tuning larger models on smaller GPUs.

**How it works.** Load NF4/quantized base; train LoRA; careful optimizer states management.

**Intuition.** Freeze a compressed brain; train the slim hat.

**Common pitfalls.**
- Quality drops if quantization too aggressive for the task
- Tooling complexity

**When to use.** When GPU memory is the bottleneck.


In [ ]:
# Demo 1 — memory sketch
params = 7e9
fp16 = params*2
int4 = params*0.5
print({"fp16_GB": fp16/1e9, "int4_GB_approx": int4/1e9})


In [ ]:
# Demo 2 — training precision note
print("base weights quantized; LoRA matrices train in fp16/bf16")


### Try it yourself — QLoRA

1. When would full 16-bit LoRA be preferred over QLoRA?


## Adapters

**Definition.** Small bottleneck modules inserted between layers (classic Houlsby/Pfeiffer adapters) trained while freezing the base.

**Why it matters.** Another PEFT family—sometimes preferred for modular multi-task setups.

**How it works.** Insert down/up projections with nonlinearity; train those params.

**Intuition.** Plugin modules between frozen layers.

**Common pitfalls.**
- Latency from extra modules if not fused
- Naming confusion with LoRA 'adapters'

**When to use.** Multi-task research systems; understand for literature.


In [ ]:
# Demo 1 — bottleneck shapes
d, b = 1024, 64
print("adapter params", 2*d*b)


In [ ]:
# Demo 2 — LoRA vs classic adapter
print({"LoRA": "low-rank delta on W", "Adapter": "extra module in block"})


### Try it yourself — Adapters

1. State one similarity and one difference vs LoRA.


## Glossary

- **LoRA**: Low-rank adaptation of frozen weights
- **QLoRA**: Quantized base + LoRA training


### Workshop drill — Model Adaptation (1)

Restate each section heading as a single exam-ready sentence.


In [ ]:
# Workshop drill 1 — Model Adaptation
headings = ['Fine-tuning', 'PEFT', 'LoRA', 'QLoRA', 'Adapters']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Model Adaptation (2)

Change one hyperparameter/assumption in a demo and predict the effect before running.


In [ ]:
# Workshop drill 2 — Model Adaptation
print('prediction: ...')
print('observation: ...')
print('delta: ...')


### Workshop drill — Model Adaptation (3)

List production risks (cost, latency, safety, quality) for this topic.


In [ ]:
# Workshop drill 3 — Model Adaptation
for r in ['cost','latency','safety','quality']:
    print(f'{r}:')


## Summary & Key Takeaways

- Prefer prompts/RAG before heavy fine-tunes for facts
- PEFT/LoRA is the practical adaptation default
- QLoRA trades precision for memory headroom
- Govern adapters like code artifacts

### Practice

Draft a go/no-go checklist for proposing LoRA at work.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
